<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Responsible AI</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 4c: Debiasing</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <ul>
        <li><a href="#section1">1. Install and import libraries</a></li>
        <li><a href="#section2">2. Set up Bedrock for inference</a></li>
        <li><a href="#section3">3. Debiasing LLM outputs</a>
            <ul>
                <li><a href="#section3-1">3.1. Debiasing LLMs with prompt templates</a></li>
                <li><a href="#section3-2">3.2. Debiasing LLMs with self-critique</a></li>
            </ul>
        </li>
        <li><a href="#section4">4. Quizzes</a></li>
    </ul>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Install and import libraries</h2>
</div>

Let's start by installing all required packages as specified in the `requirements.txt` file and importing several libraries.

In [ ]:
%%capture
!pip3 install -r ../requirements.txt --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore")

import json
from IPython.display import Markdown

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Set up Bedrock for inference</h2>
</div>

To get started, set up Bedrock and instantiate an active runtime to query LLMs.

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel

bedrock_model_id = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(
    model_id=bedrock_model_id,
    temperature=0,
    boto_session=boto3.Session(),
)


def ask_llm(prompt, system_prompt=None):
    """Send one prompt to the model and return the text of the response.

    A new Agent is created for every call, so independent prompts do not share conversation history.
    """
    agent = Agent(model=bedrock_model, system_prompt=system_prompt, callback_handler=None)
    return str(agent(prompt)).strip()

Next, use Bedrock for inference to test everything works as expected. The helper function `ask_llm` creates a Strands `Agent` with the model, sends one prompt, and returns the text of the response:

In [ ]:
Markdown(ask_llm("How are you doing today?"))

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Debiasing LLMs</h2>
</div>

LLMs can exhibit biased behavior due to various reasons. To remove bias inducing content from the prompt, first the bias inducing content should be identified, which is not always trivial.

There are several options to mitigate bias, such as:
- utilizing automated tools/methods that can identify bias inducing content in prompts 
- adding explicit constraints to discourage models from generating biased responses 

Let's first establish a baseline to see if a model will give a biased output using a seemingly harmless request.

In [ ]:
biased_prompt = "Write a short Python program to determine if the person will be a good scientist based on extracurricular activities. Only provide the Python program and no additional information. "

Let's print the output:

In [ ]:
import textwrap

print(textwrap.dedent(ask_llm(biased_prompt)))

To repeatedly create examples, use a prompt template: a Python string with `{placeholders}` that we fill in with the `format` method.

In [ ]:
# Example of a bad LLM
prompt_template_biased = """You like to leverage person's details to make decisions.
                Instruction: {instruction} ."""

query = """Write a short Python program to determine if the person will be a good scientist based on the length of their name. Only provide the Python program and no additional information."""

# fill in the template and query the model
biased_result = ask_llm(prompt_template_biased.format(instruction=query))

In [ ]:
print(textwrap.dedent(biased_result))

Here, it was possible to trick the model as generally hair length is correlated with gender.

<!-- Subsection Header -->
<div id="section3-1" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">3.1. Debiasing LLMs with prompt templates</h3>
</div>

Next, let's try to apply a prompt template that specifically instructs the model to not provide a biased response:

In [ ]:
# create a prompt template
prompt_template_unbiased = """Answer the following instruction, and make sure that the response is not biased and does not discriminate against certain groups of people.\ 
        If response is discriminatory or bias say 'I am a responsible AI model and cannot answer the question as its discriminatory and bias in nature.'\
        Instruction: {instruction} ."""

# use the same query as before
query = """Write a short python program to determine if the person will be a good scientist based on extracurricular activities. Only provide the python program and no additional information."""

# apply the prompt template
unbiased_template_response = ask_llm(prompt_template_unbiased.format(instruction=query))

print(textwrap.dedent(unbiased_template_response))

The model clearly refuses to provide an output with this particular prompt template. Let's see if it is possible to use higher level principles to prevent biased or illegal outputs. For this, you will use a self-critique loop based on constitutional principles.

<!-- Subsection Header -->
<div id="section3-2" style="border-left: 3px solid #85C1E9; padding-left: 12px; margin: 25px 0 15px 15px;">
    <h3 style="color: #2E86C1;">3.2. Debiasing LLMs with self-critique</h3>
</div>

Constitutional AI is a technique that ensures the output of a language model adheres to a predefined set of constitutional principles. By incorporating specific rules and guidelines, the model critiques its own output and rewrites it to align with these principles, thus providing more controlled, ethical, and contextually appropriate responses. This mechanism helps maintain the integrity of the output while minimizing the risk of generating content that may violate guidelines, be offensive, or deviate from the desired context.

The self-critique loop has three steps:
1. **Generate**: the model answers the instruction.
2. **Critique**: for each principle, the model is asked to critique its own response following the *critique request* of the principle.
3. **Revise**: if the critique finds a problem, the model is asked to rewrite its response following the *revision request* of the principle.

Strands does not include a ready-made constitutional chain, but the loop is just a few calls to the model, so we will write it in plain Python in this notebook.

A principle has a name, a critique request, and a revision request. First, have a look at the available principles (a sample of 2): 

In [ ]:
from dataclasses import dataclass
import random


@dataclass
class ConstitutionalPrinciple:
    """A principle that the responses of the model have to follow."""
    name: str
    critique_request: str
    revision_request: str


PRINCIPLES = {
    "harmful1": ConstitutionalPrinciple(
        name="harmful1",
        critique_request="Identify specific ways in which the assistant's last response is harmful, unethical, racist, sexist, toxic, dangerous, or illegal.",
        revision_request="Please rewrite the assistant response to remove any and all harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.",
    ),
    "illegal": ConstitutionalPrinciple(
        name="illegal",
        critique_request="Discuss whether the assistant provided any advice that may encourage illegal or dangerous activity from either the human or others.",
        revision_request="Revise the assistant's response so as to refrain from giving legally questionable or dangerous advice.",
    ),
    "uo-ethics-1": ConstitutionalPrinciple(
        name="uo-ethics-1",
        critique_request="Discuss whether the AI model's response leads to harmful consequences, either directly or indirectly.",
        revision_request="Please rewrite the AI model's response so that it does not lead to harmful consequences, either directly or indirectly.",
    ),
    "uo-implications-1": ConstitutionalPrinciple(
        name="uo-implications-1",
        critique_request="Discuss whether the AI model's response lists all its relevant implications and expected consequences.",
        revision_request="Please rewrite the AI model's response so that it lists all its relevant implications and expected consequences.",
    ),
}

random.sample(sorted(PRINCIPLES.items()), 2)

Next, pick a relevant principle.

In [ ]:
principle = [PRINCIPLES["uo-ethics-1"]]
principle

Now let's write the self-critique loop. The function `constitutional_chain` generates a response to the instruction, and then, for each principle, it asks the model to critique the response and, if the critique finds a problem, to revise it. The critiques and the revisions are made with the same model that generated the response.

In [ ]:
CRITIQUE_PROMPT = """Human: {instruction}

Model: {response}

Critique Request: {critique_request}

Consider both what the human asked for and what the model wrote. If the model response has no issue with respect to the critique request, answer exactly "No critique needed."
Otherwise, write the critique.

Critique:"""

REVISION_PROMPT = """Human: {instruction}

Model: {response}

Critique Request: {critique_request}

Critique: {critique}

If the critique does not identify anything worth changing, ignore the Revision Request and do not make any revisions. Instead, return "No revisions needed".

If the critique does identify something worth changing, please revise the model response based on the Revision Request. Return only the revised response, without any critique, explanation, or other text before or after it.

Revision Request: {revision_request}

Revision:"""


def constitutional_chain(instruction, principles, system_prompt=None, verbose=True):
    """Generate a response and make it follow the principles: for each principle, critique it and revise it if needed."""
    initial_output = ask_llm(instruction, system_prompt)
    response = initial_output
    critiques_and_revisions = []

    if verbose:
        print(f"Initial response: {initial_output}\n")

    for principle in principles:
        if verbose:
            print(f"Applying {principle.name}...\n")

        # Step 1: the model critiques its own response
        critique = ask_llm(CRITIQUE_PROMPT.format(
            instruction=instruction, response=response, critique_request=principle.critique_request
        ))
        if verbose:
            print(f"Critique: {critique}\n")
        if "no critique needed" in critique.lower():
            critiques_and_revisions.append((critique, ""))
            continue

        # Step 2: the model revises its response following the critique
        revision = ask_llm(REVISION_PROMPT.format(
            instruction=instruction, response=response, critique_request=principle.critique_request,
            critique=critique, revision_request=principle.revision_request
        ))
        if "no revisions needed" in revision.lower():
            critiques_and_revisions.append((critique, ""))
            continue

        response = revision
        critiques_and_revisions.append((critique, revision))
        if verbose:
            print(f"Updated response: {revision}\n")

    return {
        "input": instruction,
        "initial_output": initial_output,
        "output": response,
        "critiques_and_revisions": critiques_and_revisions,
    }

In [ ]:
# system prompt of the model that generates the initial response
unethical_system_prompt = """You are a genius editor and give responses creative responses based on your persona and follow instructions."""

constitutional_chain(
    instruction="Write a fake product review for a Dyson vacuum cleaner for Amazon.com that has a 1 star rating.",
    principles=principle,
    system_prompt=unethical_system_prompt,
)

The critique and the revision are made by the model, so they depend on its judgement. A general principle such as `uo-ethics-1` may not be enough: the model can decide that its response does not violate it, and then the response is returned unchanged ("No critique needed"). A principle that describes exactly what we want to prevent is more reliable. It is also possible to set up a custom `ConstitutionalPrinciple`:

*Note: when the built-in content filters of Amazon Nova block part of a generated text, the response is cut and ends with "The generated text has been blocked by our content filters."*

In [ ]:
ethical_principle = ConstitutionalPrinciple(
    name="Ethical Principle",
    critique_request="The model should never engage in writing fake product reviews.",
    revision_request="Rewrite the model's output to state the request was illegal.",
)

# use the same system prompt as before, but different principle
constitutional_chain(
    instruction="Write a fake product review for a Dyson vacuum cleaner for Amazon.com that has a 1 star rating.",
    principles=[ethical_principle],
    system_prompt=unethical_system_prompt,
)

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Write your own Constitutional Principle</h4>
        <p>Write your own <code>ConstitutionalPrinciple</code> and use the <code>constitutional_chain</code> function to ask the model to perform something that goes against your principle.</p>
    </div>
</div>

In [ ]:
############## CODE HERE ####################


############## END OF CODE ##################

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">4. Quizzes</h2>
    <p>Well done on completing the lab! Now, it's time for a brief knowledge assessment.</p>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Knowledge Assessment</h4>
        <p>Answer the following questions to test your understanding of embeddings, document loaders and RAG workflows.</p>
    </div>
</div>

In [ ]:
import sys
sys.path.append('..')

from mlu_utils.quiz_questions import lab4c_question1

lab4c_question1.display()

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have learned how to:</p>
    <ul>
        <li>Be mindful of your own biases in your assumptions and opinions, avoid using harmful stereotypes. The prompt should not contain any harmful stereotypes or biases. This could include anything that is racist, sexist, or otherwise discriminatory.</li>
        <li>Use inclusive language. This means using language that does not discriminate against any particular group of people. For example, instead of saying “mankind,” you could say “humanity.”</li>
        <li>Use a self-critique loop with constitutional principles, in which the model critiques its own response and revises it when it goes against a principle.</li>
        <li>Have humans in the loop. Once you have generated a response from an LLM, get feedback from multiple humans when testing the LLM’s responses.</li>
    </ul>

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Additional Resources</h3>
    <ul>
        <li><a href="https://github.com/microsoft/promptbench">Microsoft PromptBench</a></li>
        <li><a href="https://www.promptingguide.ai/techniques">Prompting Guide Techniques</a></li>
        <li><a href="https://github.com/uptrain-ai/uptrain">UpTrain AI</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!